In [2]:
%uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%uv pip install transformers datasets accelerate scikit-learn grapheme


Using Python 3.12.6 environment at: /usr/local
Audited 3 packages in 27ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 5 packages in 25ms
Note: you may need to restart the kernel to use updated packages.


In [3]:
%uv pip install torch torchvision torchaudio
%uv pip install transformers datasets accelerate scikit-learn grapheme


Using Python 3.12.6 environment at: /usr/local
Audited 3 packages in 8ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 5 packages in 27ms
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
df = pd.read_csv("trainV2.csv")  
df.head()

,Text,Class
0,நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...,Non-Abusive
1,உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...,Abusive
2,கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...,Non-Abusive
3,ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...,Abusive
4,ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...,Non-Abusive


In [6]:
import re, html, unicodedata
import grapheme

def grapheme_preprocess(text):
    text = str(text)

    # decode html
    text = html.unescape(text)

    # normalize unicode (VERY IMPORTANT)
    text = unicodedata.normalize("NFC", text)

    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)

    # extract grapheme clusters
    clusters = list(grapheme.graphemes(text))

    # rejoin normalized graphemes
    text = "".join(clusters)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [7]:
df["grapheme_text"] = df["Text"].apply(grapheme_preprocess)
df["clean_text"] = df["Text"]  # baseline version


In [21]:
TEXT_COL = "grapheme_text"
#TEXT_COL = "grapheme_text" for baseline version

In [22]:
print(df.columns)


Index(['Text', 'Class', 'grapheme_text', 'clean_text', 'label'], dtype='object')


In [23]:
df["label"] = df["Class"].map({
    "Non-Abusive":0,
    "Abusive":1
})


In [24]:
# normalize label text
df["Class"] = df["Class"].astype(str).str.strip().str.lower()

print("Unique labels after cleaning:", df["Class"].unique())

# map to numeric
df["label"] = df["Class"].map({
    "abusive":1,
    "non-abusive":0,
    "non abusive":0
})

# check if any missing
print("NaN labels count:", df["label"].isna().sum())

Unique labels after cleaning: ['non-abusive' 'abusive']
NaN labels count: 0


In [25]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df[[TEXT_COL,"label"]],
    test_size=0.1,
    stratify=df["label"],
    random_state=42
)


In [26]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df.rename(columns={TEXT_COL:"text"})
)
val_dataset = Dataset.from_pandas(
    val_df.rename(columns={TEXT_COL:"text"})
)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])
val_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])


Map:   0%|          | 0/3286 [00:00<?, ? examples/s]

Map:   0%|          | 0/366 [00:00<?, ? examples/s]

In [28]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,

    eval_strategy="epoch",   # ← use THIS (not eval_strategy)
    save_strategy="epoch",

    load_best_model_at_end=True,
    report_to=None
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


trainer.train()


Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.575983,0.762295,0.741840
2,No log,0.536426,0.748634,0.756614
3,0.555900,0.519555,0.778689,0.781671
4,0.555900,0.502316,0.797814,0.795580
5,0.340600,0.514277,0.795082,0.795640


TrainOutput(global_step=1030, training_loss=0.44454829114154704, metrics={'train_runtime': 386.87, 'train_samples_per_second': 42.469, 'train_steps_per_second': 2.662, 'total_flos': 1080728659891200.0, 'train_loss': 0.44454829114154704, 'epoch': 5.0})

In [34]:
trainer.evaluate()

{'eval_loss': 0.5023160576820374,
 'eval_accuracy': 0.7978142076502732,
 'eval_f1': 0.7955801104972375,
 'eval_runtime': 2.3709,
 'eval_samples_per_second': 154.373,
 'eval_steps_per_second': 9.701,
 'epoch': 5.0}

In [35]:
trainer.save_model("best_model_grapheme")
tokenizer.save_pretrained("best_model_grapheme")

('best_model_grapheme/tokenizer_config.json',
 'best_model_grapheme/special_tokens_map.json',
 'best_model_grapheme/vocab.txt',
 'best_model_grapheme/added_tokens.json',
 'best_model_grapheme/tokenizer.json')

In [36]:
import shutil
shutil.make_archive("murel_grapheme_best_model", 'zip', "best_model_grapheme")


'/root/murel_grapheme_best_model.zip'

In [20]:
shutil.make_archive("murel_clean_results", 'zip', "results")

'/root/murel_clean_results.zip'

In [ ]:
from google.colab import files
files.download("best_model.zip")
